# Quantum Fourier Transform, and SHor's algorithm

Matthew Thompson

Eastern Washington University

Math-481, Prof. Y. Nievergelt

6/9/2026

## **Introduction**




### **Introduction**

The fourier transform takes a continuous real-valued function $f(x)$ and converts it to a complex-valued function $\hat{f}(\xi)$ in frequency basis, by

\begin{align*} \hat{f}(\xi) = \int _{-\infty}^{\infty} f(x) e^{-i2 \pi x \xi} d\xi. \end{align*}

The discrete version of the Fourier transform (DFT) maps a sequence of $N$ complex numbers $x_n$ to another sequence $\hat{x}_k$:

\begin{align*} \hat{x}_k = \frac{1}{\sqrt{N}} \sum_{n=0}^{N-1} x_n e^{-i 2 \pi k n / N}. \end{align*}

The DFT and its various FFT algorithms are of primary importance accross mathematics and the applied sciences.

So too, does the DFT find usages in quantum computing algorithms. One example is Shor's algorithm, which factorizes a number $N$ into prime factors in polynomial in $\log N$ time. This is remarkably faster than the best classical algorithms, which run in sub-exponential time.

The purpose of this report is to  document of the author's study of the Quantum Fourier Transform (QFT), and to give an explanation of Shor's algorithm. Shor's algorithm factorizes integers by using Quantum Phase Estimation (QPE), (which relies on QFT), and a bit of number theory and spectral theory.




## **DFT (Discrete Fourier Transform)**

### **Hilbert space**

Hilbert spaces are vector spaces, in that they are closed under addition between elements and scalar multiplication by elements of a field (in this paper, $\mathbb{C}$), and have a zero vector.

They also have additional features:

- Hilbert spaces have an inner product defined between vectors. 

- Hilbert spaces are complete, meaning all Cauchy sequences of vectors in the Hilbert space converge to a limit within that space. 

- Hilbert spaces need not be finite-dimesnional. (However for our purposes, we will only deal with finite-dimensional spaces.)

### **Inner product**
We define an inner product between vectors $x = (x_0, x_1, \cdots, x_{n-1})^T$ and $y = (y_0, y_1, \cdots, y_{n-1})^T$ as

\begin{align*} \langle x, y \rangle = \sum_{i=0}^{n-1} \overline{x_i} y_i . \end{align*}

That is, the inner product is antilinear in the first argument. That means that for $\alpha \in \mathbb{C}$ with complex conjugate $\overline{\alpha}$, 

\begin{align*} \alpha \langle x, y \rangle = \langle \overline{\alpha}x, y \rangle  = \langle x, \alpha y \rangle \end{align*}.

With this inner product we have an associated norm,

\begin{align*} ||x||^2 =  \langle x, x \rangle. \end{align*}

### **Orthogonality**

Two non-zero vectors $x, y$ are orthogonal if and only if

\begin{align*} \langle x, y \rangle = 0 \end{align*}

### **Roots of unity**

<!-- Roots of unity are complex numbers of the form 

\begin{align*} z = e^{k \frac{2 \pi i}{n}} \end{align*}

where $k \in \{0, 1, \cdots, n-1\}$, such that $z^n = 1$. 

Specifically, $e^{k \frac{2 \pi i}{n}}$ are called the $n$-th roots of unity, and $e^{\frac{2 \pi i}{n}}$ is called the primitive $n$-th root of unity. 

We will soon encounter fourier basis vectors, which are vectors with all the roots of unity as coefficients. 

These fourier basis vectors are orthogonal for distinct $n \neq m$.

\begin{align*} \langle e^{k \frac{2 \pi i}{n}} \vert e^{k \frac{2 \pi i}{m}} \rangle = \cdots = 0 \end{align*}

This will become quite relevant when dealing with the QFT, because it implies that $\cdots$ -->

The $n$-th roots of unity are the $k$ complex numbers satisfying $z^n = 1$, given by

\begin{align*}
    e^{\frac{2\pi i k}{n}}, \qquad k \in \{0, 1, \ldots, n-1\}.
\end{align*}
They are evenly spaced on the unit circle in $\mathbb{C}$. The value $ e^{\frac{2\pi i}{n}}$ is called the primitive $n$-th root of unity, and every other $n$-th root of unity is a power of it.

and note that equivalently, the set of $n$-th roots of unity is $\{ e^{\frac{-2\pi i k}{n}}\; : \; k \in \{0, 1, \ldots, n-1\}\}$.


### **Overview of DFT**

Denote $\omega = e^{\frac{-2\pi i}{N}}$, and $\omega^k = e^{\frac{-2\pi i k}{N}}$.

For the Discrete Fourier Transform, we will take a discrete sample of the signal $f(x)$, which can be interpreted as a vector $x = (x_0, x_1, \cdots, x_{N-1})$ . Then $x$ is mapped to $\hat{x}$ , where $ \hat{x} = (\hat{x}_0, \cdots \hat{x}_{N-1})$, where
$$ \hat{x}_k = \frac{1}{\sqrt{N}}\sum_{l=0} ^{N-1} x_l \omega^{kl}.$$

This represents a change of basis from the standard basis $\{e_0, e_1, ... ,e_{N-1}\}$ to the fourier basis $\{ \phi_0, \phi_1, \cdots, \phi_{N-1}\}$ where $$\phi_k = \frac{1}{\sqrt{N}}(1, \omega^{k}, \omega^{2k} \cdots, \omega^{(N-1)k}).$$



(The DFT can be computed more efficiently using the FFT, and a specific example of an FFT algorithm is the Cooley-Tukey algorithm. See \cite{Barros2026}.)

The DFT is a linear operation, and thus can be represented by a matrix, $F_N$.

If we denote the $N^\text{th}$ roots of unity as $\omega := e^{\frac{-2 \pi i}{N}}$, then this transformation matrix is $F_{N} = \frac{1}{\sqrt{N}}\left(\omega^{jk}\right)_{j,k = 0, \cdots, N-1}$.

We observe that $F_N$ is a symmetric matrix. 





### **Orthogonality of fourier bases**
Distinct fourier bases are orthogonal (and in fact, orthonormal). This follows from a property of the $N$-th roots of unity: 

Given any integer $m$,

\begin{align*}
    \frac{1}{N}\sum_{k=0}^{N-1} \omega^{mk} = \begin{cases} 1 & \text{if } N \mid m \\ 0 & \text{otherwise.} \end{cases}
\end{align*}

$\textbf{Proof:}$ If $N \mid m$, then $m = Nj$ for some integer $j$, so

\begin{align*}
    \omega^m = e^{-\frac{2\pi i m}{N}} = e^{-\frac{2\pi i Nj}{N}} = e^{-2\pi i j} = 1.
\end{align*}

Thus every term in the sum equals 1, and

\begin{align*}
    \frac{1}{N}\sum_{k=0}^{N-1} \omega^{mk} = \frac{1}{N}\sum_{k=0}^{N-1} 1 = \frac{N}{N} = 1.
\end{align*}

If $N \nmid m$, the ratio $r = \omega^m \neq 1$, and by the geometric series formula,

\begin{align*}
    \frac{1}{N}\sum_{k=0}^{N-1} r^k = \frac{1}{N}\cdot\frac{r^N - 1}{r - 1} = \frac{1}{N}\cdot\frac{e^{-2\pi i m} - 1}{r-1} = \frac{1}{N}\cdot\frac{0}{r-1} = 0.
\end{align*}


### **Unitary operator**

(Let $U^\dagger$ denote the adjoint, wich is the complex conjugate of the transpose of the matrix $U$.)

Unitary operators have the property that

\begin{align*} U U^\dagger = U^\dagger U = I \end{align*},

or in other words,

\begin{align*} U^\dagger = U^{-1}. \end{align*}  

From this it follows that unitary matrices are norm-preserving, i.e.

\begin{align*} || \vert \psi  \rangle || = ||U \vert \psi \rangle||. \end{align*}

$\textbf{Proof:}$

\begin{align*}
    \|U\vert\psi\rangle\|^2 
    = \langle U\psi, U\psi\rangle 
    = \langle \psi, U^\dagger U\psi\rangle 
    = \langle \psi, \psi\rangle 
    = \|\vert\psi\rangle\|^2 \\
    \implies \|U\vert\psi\rangle\| = \|\vert\psi\rangle\|. 
\end{align*}




### **Eigenvalues of unitary operators**

Eigenvalues of unitary operators have a special property:

For a unitary operator $U$ with eigenvector $\vert \psi \rangle$ such that

\begin{align} U \vert \psi \rangle = \lambda \vert \psi \rangle, \end{align}

the eigenvalue $\lambda$ has magnitude $| \lambda | = 1$, or in other words $\lambda = e^{i \theta}$ for some phase $\theta$.

$\textbf{Proof:}$ Since $U$ is unitary it is norm-preserving, so

\begin{align*}
    \|\vert\psi\rangle\| = \|U\vert\psi\rangle\| = \|\lambda\vert\psi\rangle\| = |\lambda|\,\|\vert\psi\rangle\|.
\end{align*}
Since $\vert\psi\rangle$ is an eigenvector it is nonzero, so we may divide both sides by $\|\vert\psi\rangle\|$ to obtain $|\lambda| = 1$. Any complex number of modulus 1 can be written as $e^{i\theta}$ for some $\theta \in [0, 2\pi)$, so $\lambda = e^{i\theta}$. 

### **DFT matrix is unitary:**

$F_n \textbf{ is a unitary operator.}$ 

\begin{align*} \langle m \vert F_q ^\dagger F_q \vert k \rangle = \delta_{mk}, \\
\implies F_q ^\dagger F_q = I. \end{align*}

$\textbf{Proof:}$ This follows from the orthogonality of the Fourier bases. (Full proof given in \cite{Barros_2026})

## **Qubits and quantum states**






### **Tensor Product**
Without a formal definition, for two vector spaces over the same field $V_1 = \text{Span}\{v_0, v_1, \cdots, v_n\}$ and $V_2 = \text{Span}\{w_0, w_1, \cdots, w_m\}$, 
for vectors $x = (x_0, \ldots, x_n)^T \in V_1$
and $y = (y_0, \ldots, y_m)^T \in V_2$​, we define the tensor product $x \otimes y$ as

\begin{align*} x \otimes y = (x_0 y_0,\ x_0 y_1,\ \ldots,\ x_0 y_m,\ x_1 y_0,\ \ldots,\ x_n y_m)^T \in V_1 \otimes V_2. \end{align*}
i.e. each element of $x$ is multiplied by every element of $y$. 
We define the tensor product between vector spaces (over the same field) as 

\begin{align*} V_1 \otimes V_2 = \text{Span}\{v_i \otimes w_j : 0 \leq i \leq n, 0 \leq j \leq m\}.\end{align*}

Thus $V_1 \otimes V_2$ has dimension $(n+1)(m+1)$, and its basis vectors are all pairwise tensor products of the basis vectors for $V_1$ and $V_2$.

### **Qubit/Quantum state**

A particularly interesting class of vectors in $\mathbb{C}^N$ are those corresponding to quantum states, which when employed in a quantum computer, are called qubits.

A classical bit takes one of two values in $\mathbb{Z}_2$. The basis states for which a qubit exactly corresponds with one of the classical bits are denoted $\vert 0 \rangle$ and $\vert 1 \rangle$. What makes a qubit unique compared to a classical bit, is that it can take on values which are complex-weighted superpositions of the two basis states corresponding to the values in $\mathbb{Z}_2$. That is, $\vert \psi \rangle = \alpha \vert 0 \rangle + \beta \vert 1 \rangle$, where $\alpha, \beta \in \mathbb{C}$. With $\vert 0 \rangle$ and $\vert 1 \rangle$ serving as orthonormal basis vectors, a single qubit takes on values in $\mathbb{C}^2$. For this paper on quantum computers, we are only concerned with qubits taking on pure states, which are superpositions $\vert \psi \rangle = \alpha \vert 0 \rangle + \beta \vert 1 \rangle$ such that $|\alpha|^2 + | \beta | ^2 = 1$.

A system with $n$ qubits resides in $\mathbb{C}^{2^n}$. This is because we take the tensor product of each qubit's basis to give the full computational basis. E.g. in a $2$-qubit system, the basis is 

\begin{align*} \{\vert 0 \rangle, \vert 1 \rangle\} \otimes \{\vert 0 \rangle, \vert 1 \rangle\} = \{\vert 00\rangle, \vert 01\rangle, \vert 10\rangle, \vert 11\rangle\}.\end{align*}

Then, any $n$-qubit state can be expressed as a linear combination of the computational basis states in $\{\vert 0 \rangle, \vert 1 \rangle\}^{\otimes n}$.






### **Computational basis** 

A single-qubit has two computational basis states. In the z-basis these are $\vert 0 \rangle$ and $\vert 1 \rangle$, (sometimes referred to as $\vert +z \rangle$ and $\vert -z \rangle$.) In a single-qubit system, these make up the computational basis. Combinations of these bases can make other pure states, such as 

\begin{align*} \vert + \rangle &= \frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}} \\
\vert - \rangle &= \frac{\vert 0 \rangle - \vert 1 \rangle}{\sqrt{2}} \\
\vert +i \rangle &= \frac{\vert 0 \rangle + i\vert 1 \rangle}{\sqrt{2}} \\
\vert -i \rangle &= \frac{\vert 0 \rangle - i\vert 1 \rangle}{\sqrt{2}} \end{align*}

(Sometimes referred to as $\vert +x \rangle$, $\vert -x \rangle$, $\vert +y \rangle$, and $\vert -y \rangle$, respectively.)

A note on notation: decimal and binary notation for the computational basis is used interchangeably. So in a hilbert space with dimension $N = 2^n$,

we could write the set of computational basis states in decimal form as $\{\vert 0 \rangle, \vert 1 \rangle, \cdots, \vert N-1 \rangle \}$ with general state

\begin{align*} \vert \psi \rangle = \sum_{l=0}^{N-1} c_l \vert l \rangle, \end{align*}

or write the computational basis in binary form as $ \{ \vert 00\cdots 01\rangle, \vert 00\cdots 10\rangle, \cdots, \vert 11 \cdots 11\rangle \}$ with general state

\begin{align*} \vert \psi \rangle = \sum_{l_0 = 0}^{1} \sum_{l_1 = 0}^{1} \cdots \sum_{l_{n-1} = 0}^{1} c_{l_0 l_1 \cdots l_{n-1}} \vert l_0 l_1 \cdots l_{n-1} \rangle. \end{align*}

The decimal representation is what will generally be used for readability, but the binary representation is nice because it illustrates that the computation basis is

\begin{align*} \text{BASIS} = \{\vert 0 \rangle, \vert 1 \rangle\}^{\otimes n}
= \{ \vert l_0 \rangle \otimes \vert l_1 \rangle \otimes \cdots \otimes \vert l_{n-1} \rangle : l_i \in \{0, 1\}\}. \end{align*}

The decimal and binary labels refer to the same basis states. For example,

\begin{align*} \lvert 5\rangle = \lvert 101\rangle. \end{align*}

The coefficients of quantum states represent probability amplitudes, and so we have $\sum |c_i|^2 = 1$ for any quantum state. (Born rule.)







### **Quantum gates** 
Quantum gates are transformations acting on one or more qubit in a quantum system. 

Note that all quantum gates are unitary. Since the coefficients of quantum states are probability amplitudes, the states have norm 1, and we want evolution of the system to preserve that norm. Also the waiting apparatus for a system governed by the Schrödinger equation is unitary, (discussed in class).

A quantum gate can be expressed as a truth table \cite{Shor}.

A general single-bit gate $G = \begin{bmatrix} g_{00} & g_{01} \\ g_{10} & g_{11} \end{bmatrix}$ 

acting on a general qubit $\vert \psi_0 \rangle = \alpha \vert 0 \rangle + \beta \vert 1 \rangle$ can be represented as


\begin{align*}
G \vert \psi_0 \rangle
&=
G(\alpha \vert 0 \rangle + \beta \vert 1 \rangle) \\
&=
\alpha G \vert 0 \rangle + \beta G \vert 1 \rangle \\
&=
\begin{bmatrix}
g_{00} & g_{01} \\
g_{10} & g_{11}
\end{bmatrix}
\begin{pmatrix}
\alpha \\
\beta
\end{pmatrix}.
\end{align*}

or as 

\begin{align*}
G \vert 0 \rangle
&=
g_{00}\vert 0 \rangle + g_{10}\vert 1 \rangle,\\
G \vert 1 \rangle
&=
g_{01}\vert 0 \rangle + g_{11}\vert 1 \rangle.
\end{align*}

#### **Hadamard operator:**
  The Hadamard gate maps each basis state to a superposition. $H: \mathbb{C}^2 \to \mathbb{C}^2$,
  
  \begin{align*} H = \frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}}\langle 0 \vert + \frac{\vert 0 \rangle - \vert 1 \rangle}{\sqrt{2}}\langle 1 \vert, \end{align*}

  Or in other words, $H$ maps 

  \begin{align*} \vert 0 \rangle &\mapsto \frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}}, \\ 
  \vert 1 \rangle &\mapsto \frac{\vert 0 \rangle - \vert 1 \rangle}{\sqrt{2}}. \end{align*}

  In matrix form,

  \begin{align*} H = \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix} \end{align*}.

#### **SWAP gate:**
  The swap gate simply swaps the states of two qubits.

  \begin{align*} \text{SWAP} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0& 0 & 1 \end{bmatrix} \end{align*}

  So that for $\vert \psi_1 \rangle = (\alpha_1, \beta_1)^T$ and $\vert \psi_2 \rangle = (\alpha_2, \beta_2)^T$, we have 

  \begin{align*} \text{SWAP}(\vert \psi_1 \rangle \otimes \vert \psi_2 \rangle) 
  = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0& 0 & 1 \end{bmatrix}
  \begin{pmatrix}\alpha_1 \alpha_2 \\ \alpha_1 \beta_2 \\ \beta_1 \alpha_2 \\ \beta_1 \beta_2 \end{pmatrix} = \begin{pmatrix}\alpha_1 \alpha_2 \\  \beta_1 \alpha_2 \\ \alpha_1 \beta_2 \\ \beta_1 \beta_2 \end{pmatrix} = \vert \psi_2 \rangle \otimes \vert \psi_1 \rangle. \end{align*}

#### **$R_k$ gates:**
$R_k$ gates correspond to a rotation of the $\vert 1 \rangle$ component of a state around the $Z$ axis of the Bloch sphere by the phase $\theta = \frac{2 \pi}{2^k}$. 

\begin{align*} R_k = \begin{bmatrix} 1 & 0 \\ 0 & e^{i\frac{2 \pi}{2^k}} \end{bmatrix} \end{align*}

(Some examples of R_k have their own names because they are important in many quantum algorithms.) 
  - $R_1 = Z$ gate
  - $R_2 = S$ gate
  - $R_3 = T$ gate


### **Bloch sphere**


A pure qubit state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ satisfies $|\alpha|^2 + |\beta|^2 = 1$, so it lives on the unit sphere in $\mathbb{C}^2 = \mathbb{R}^4$. However, two states that differ only by a global phase $e^{i\phi}$ are physically indistinguishable, so we identify $|\psi\rangle \sim e^{i\phi}|\psi\rangle$. After quotienting out this equivalence, the state space of a single qubit reduces from a 3-sphere in $\mathbb{R}^4$ down to an ordinary 2-sphere, called the Bloch sphere.

We can understand the BLoch sphere in terms of the Riemann sphere, by writing $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ with $\alpha$ real and nonnegative, so the state is parametrized by the ratio $z = \alpha/\beta \in \mathbb{C} \cup \{\infty\}$. Stereographic projection then maps this $z$ to a point on $S^2$, exactly as it does for the Riemann sphere. The north pole corresponds to $|0\rangle$ and the south pole to $|1\rangle$.

Under this picture, unitary gates act on $|\psi\rangle$ as rotations of the Bloch sphere. In particular, the $R_k$ gates defined earlier are rotations about the $Z$-axis by angle $\frac{2\pi}{2^k}$, and the Hadamard gate is a rotation by $\pi$ about the axis halfway between $X$ and $Z$.

## **QFT (Quantum Fourier Transform)**

### **In broad strokes**

The QFT is the DFT applied to a quantum state. Since $F_N$ is linear, it suffices to define it on computational basis states. Its action on a basis state $\vert j \rangle$ is

\begin{align*}
F_N \vert j \rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} \vert k \rangle,
\end{align*}

and then $F_N \vert \psi \rangle = \sum_j c_j F_N \vert j \rangle$ for a general state $\vert \psi \rangle = \sum_j c_j \vert j \rangle$. That is, the output amplitudes are the DFT of the input amplitudes.

The key structural fact is that for $N = 2^n$, the output factorizes as a tensor product. Writing $j = j_1 j_2 \cdots j_n$ in binary (so $j = j_1 2^{n-1} + j_2 2^{n-2} + \cdots + j_n$, with $j_1$ the most significant bit), and using the binary fraction notation $0.j_a j_{a+1} \cdots j_b := \frac{j_a}{2} + \frac{j_{a+1}}{4} + \cdots + \frac{j_b}{2^{b-a+1}}$, the product representation is \cite{Barros2026}

\begin{align*}
F_{2^n}\vert j_1 j_2 \cdots j_n\rangle = \frac{1}{\sqrt{2^n}}
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_n}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_{n-1}j_n}\vert 1\rangle\Bigr)
\otimes \cdots \otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_1 j_2 \cdots j_n}\vert 1\rangle\Bigr).
\end{align*}

Each tensor factor is a single-qubit state, with each successive factor encoding one more bit of $j$ in its phase. For $n = 3$ this is explicitly

\begin{align*}
F_8\vert j_1 j_2 j_3\rangle = \frac{1}{2\sqrt{2}}
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_3}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_2 j_3}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_1 j_2 j_3}\vert 1\rangle\Bigr).
\end{align*}

This product structure is what makes an efficient quantum circuit possible: each tensor factor involves only a single output qubit, so it can be prepared by a Hadamard gate followed by a small number of $C(R_k)$ gates, rather than manipulating all $2^n$ amplitudes simultaneously.### **In broad strokes**

The QFT is the DFT applied to a quantum state. Since $F_N$ is linear, it suffices to define it on computational basis states. Its action on a basis state $\vert j \rangle$ is

\begin{align*}
F_N \vert j \rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} \vert k \rangle,
\end{align*}

and then $F_N \vert \psi \rangle = \sum_j c_j F_N \vert j \rangle$ for a general state $\vert \psi \rangle = \sum_j c_j \vert j \rangle$. That is, the output amplitudes are the DFT of the input amplitudes.

For $N = 2^n$, the output factorizes as a tensor product. Writing $j = j_1 j_2 \cdots j_n$ in binary (so $j = j_1 2^{n-1} + j_2 2^{n-2} + \cdots + j_n$, with $j_1$ the most significant bit), and using the binary fraction notation $0.j_a j_{a+1} \cdots j_b := \frac{j_a}{2} + \frac{j_{a+1}}{4} + \cdots + \frac{j_b}{2^{b-a+1}}$, the product representation is \cite{Barros2026}

\begin{align*}
F_{2^n}\vert j_1 j_2 \cdots j_n\rangle = \frac{1}{\sqrt{2^n}}
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_n}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_{n-1}j_n}\vert 1\rangle\Bigr)
\otimes \cdots \otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_1 j_2 \cdots j_n}\vert 1\rangle\Bigr).
\end{align*}

Each tensor factor is a single-qubit state, with each successive factor encoding one more bit of $j$ in its phase. For $n = 3$ this is explicitly

\begin{align*}
F_8\vert j_1 j_2 j_3\rangle = \frac{1}{2\sqrt{2}}
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_3}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_2 j_3}\vert 1\rangle\Bigr)
\otimes
\Bigl(\vert 0\rangle + e^{2\pi i\, 0.j_1 j_2 j_3}\vert 1\rangle\Bigr).
\end{align*}

Each tensor factor involves only a single output qubit, so it can be prepared by a Hadamard gate followed by a small number of $C(R_k)$ gates, rather than manipulating all $2^n$ amplitudes simultaneously.

### **Roots of unity, revisited**

The QFT basis vectors $\phi_k = \frac{1}{\sqrt{N}}(1, \omega^k, \omega^{2k}, \ldots, \omega^{(N-1)k})$ have the $N$-th roots of unity as their components. The orthogonality of these basis vectors — which is what makes $F_N$ invertible — is precisely the orthogonality of roots of unity proved earlier: distinct basis vectors $\phi_k$ and $\phi_{k'}$ satisfy $\langle \phi_k, \phi_{k'} \rangle = \frac{1}{N}\sum_{j=0}^{N-1} \omega^{(k'-k)j} = 0$ whenever $k \neq k'$. The geometry of the unit circle is thus the reason the QFT is well-defined as a change of basis.

### **QFT is a unitary operator**

$F_N$ is still unitary, even when applied to quantum states. Its application corresponds to rotations of each qubit around their respective Bloch spheres.

### **Quantum circuit for QFT**

\usegraphics

## **QPE (Quantum Phase estimation)**




### **Spectral theorem for unitary matrices**

Every unitary matrix $U$ is unitarily diagonalizable: there exists a unitary $V$ such that $V^\dagger U V = \text{diag}(\lambda_0, \lambda_1, \ldots)$. Since all eigenvalues of a unitary operator have modulus 1 (proved above), they are of the form $\lambda = e^{2\pi i \theta}$ for some $\theta \in [0, 1)$. That is, the eigenvalues of any unitary matrix lie on the unit circle in $\mathbb{C}$.

QPE can therefore be understood as computing $\text{Arg}(\lambda)$ for an eigenvalue $\lambda$ of $U$: given access to $U$ and an eigenvector $\vert \psi \rangle$, QPE estimates the phase $\theta$ such that $U\vert\psi\rangle = e^{2\pi i \theta}\vert\psi\rangle$.

### **Controlled-U gates**

A controlled-U gate selectively applies some gate $U$ to one qubit (the target qubit), dependent on the state of another (the control qubit). If the control qubit is $\vert 0 \rangle$, nothing happens and if the control qubit is $\vert 1 \rangle$, $U$ is applied. Thus the controled-U gate applied to $\vert \phi_1 \rangle$ controlled by $\vert \phi_0 \rangle$ is a 4x4 matrix which operates on $\vert \phi_0 \rangle \otimes \vert \phi_1 \rangle$, given by 

\begin{align*} C(U) = \begin{bmatrix} I_2 & 0 \\ 0 & U \end{bmatrix}. \end{align*}

Depending on wether $\vert \phi_0 \rangle$ $= \vert 0 \rangle$ or $\vert 1 \rangle$, $C(U)$ maps 

\begin{align*} \vert 0 \rangle \vert \phi_1 \rangle &\mapsto \vert 0 \rangle \vert \phi_1 \rangle, \\
\vert 1 \rangle \vert \phi_1 \rangle &\mapsto \vert 1 \rangle U \vert \phi_1 \rangle. \end{align*}

Relevant to QPE are the controlled gates 

\begin{align*} \text{CNOT} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix} \end{align*}

\begin{align*} C(R_k) = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & e^{\frac{2 \pi i}{2^k}} \end{bmatrix} \end{align*}

### **Phase kickback**

In the controlled-$U$ gate explanation above, we assumed the control qubit was a basis state $\vert 0 \rangle$ or $\vert 1 \rangle$. Now suppose instead the control qubit is in superposition, and the target is an eigenvector $\vert \psi \rangle$ of $U$ with eigenvalue $e^{2\pi i \theta}$. The initial state is

### **Phase kickback**

In the controlled-$U$ gate discussion above, we assumed the control qubit was a basis state $\vert 0 \rangle$ or $\vert 1 \rangle$. Now suppose instead the control qubit is in superposition, and the target is an eigenvector $\vert \psi \rangle$ of $U$ with eigenvalue $e^{2\pi i \theta}$. The initial joint state is

\begin{align*}
\frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle.
\end{align*}

Applying $C(U)$ to this state, and using linearity,

\begin{align*}
C(U)\left(\frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle\right)
&= \frac{1}{\sqrt{2}}\left(\vert 0 \rangle \vert \psi \rangle + \vert 1 \rangle U\vert \psi \rangle\right) \\
&= \frac{1}{\sqrt{2}}\left(\vert 0 \rangle \vert \psi \rangle + e^{2\pi i \theta}\vert 1 \rangle \vert \psi \rangle\right) \\
&= \frac{\vert 0 \rangle + e^{2\pi i \theta}\vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle.
\end{align*}

The phase $e^{2\pi i \theta}$ has been imparted onto the control qubit, while the target $\vert \psi \rangle$ is left unchanged. This is phase kickback.

In QPE we exploit this repeatedly. The $k$-th ancilla qubit controls $U^{2^k}$, and since $U^{2^k}\vert\psi\rangle = e^{2\pi i \cdot 2^k\theta}\vert\psi\rangle$, phase kickback imparts the phase $e^{2\pi i \cdot 2^k \theta}$ onto that qubit. After all $t$ controlled gates have been applied, the ancilla register is in the state

\begin{align*}
\frac{1}{\sqrt{2^t}}\Bigl(\vert 0 \rangle + e^{2\pi i \cdot 2^{t-1}\theta}\vert 1 \rangle\Bigr)
\otimes \cdots \otimes
\Bigl(\vert 0 \rangle + e^{2\pi i \cdot 2\theta}\vert 1 \rangle\Bigr)
\otimes
\Bigl(\vert 0 \rangle + e^{2\pi i \theta}\vert 1 \rangle\Bigr),
\end{align*}

and the second register is still $\vert \psi \rangle$. Comparing with the product representation of the QFT, this ancilla state is exactly $F_{2^t}\vert \tilde{\theta} \rangle$, where $\tilde{\theta}$ is the $t$-bit binary representation of $\theta$. Applying $F_{2^t}^\dagger$ to the ancilla and measuring therefore recovers $\theta$.\begin{align*}
\frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle.
\end{align*}

Applying $C(U)$ to this state, and using linearity,

\begin{align*}
C(U)\left(\frac{\vert 0 \rangle + \vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle\right)
&= \frac{1}{\sqrt{2}}\left(\vert 0 \rangle \vert \psi \rangle + \vert 1 \rangle U\vert \psi \rangle\right) \\
&= \frac{1}{\sqrt{2}}\left(\vert 0 \rangle \vert \psi \rangle + e^{2\pi i \theta}\vert 1 \rangle \vert \psi \rangle\right) \\
&= \frac{\vert 0 \rangle + e^{2\pi i \theta}\vert 1 \rangle}{\sqrt{2}} \otimes \vert \psi \rangle.
\end{align*}

The phase $e^{2\pi i \theta}$ has been imparted onto the control qubit, while the target $\vert \psi \rangle$ is left unchanged. This is phase kickback.

In QPE we exploit this repeatedly. The $k$-th ancilla qubit controls $U^{2^k}$, and since $U^{2^k}\vert\psi\rangle = e^{2\pi i \cdot 2^k\theta}\vert\psi\rangle$, phase kickback imparts the phase $e^{2\pi i \cdot 2^k \theta}$ onto that qubit. After all $t$ controlled gates have been applied, the ancilla register is in the state

\begin{align*}
\frac{1}{\sqrt{2^t}}\Bigl(\vert 0 \rangle + e^{2\pi i \cdot 2^{t-1}\theta}\vert 1 \rangle\Bigr)
\otimes \cdots \otimes
\Bigl(\vert 0 \rangle + e^{2\pi i \cdot 2\theta}\vert 1 \rangle\Bigr)
\otimes
\Bigl(\vert 0 \rangle + e^{2\pi i \theta}\vert 1 \rangle\Bigr),
\end{align*}

and the second register is still $\vert \psi \rangle$. Comparing with the product representation of the QFT, this ancilla state is exactly $F_{2^t}\vert \tilde{\theta} \rangle$, where $\tilde{\theta}$ is the $t$-bit binary representation of $\theta$. Applying $F_{2^t}^\dagger$ to the ancilla and measuring therefore recovers $\theta$.

### **QPE problem statement**

Suppose we have a unitary matrix $U$ with eigenvector $\vert \psi \rangle$, so that

\begin{align*}
U \vert \psi \rangle = e^{2\pi i \theta} \vert \psi \rangle
\end{align*}

for some unknown $\theta \in [0,1)$. The goal of QPE is to estimate $\theta$.

The algorithm uses two registers. The second register holds $\vert \psi \rangle$. The first register (the ancilla) consists of $t$ qubits each initialized to $\vert 0 \rangle$, then put into superposition by a Hadamard gate. Controlled-$U^{2^k}$ gates are applied to the second register, controlled by each ancilla qubit in turn; phase kickback causes the phase $\theta$ to be encoded into the ancilla register. Finally, the inverse QFT $F_N^\dagger$ is applied to the ancilla, and measuring it yields a $t$-bit binary approximation to $\theta$. The more ancilla qubits used, the more accurate the approximation.

### **Quantum circuit for QPE**

\usegraphics

## **Shor's Algorithm**



### **The period-finding problem**

Shor's algorithm uses period finding and a little bit of number theory. Given $a$ coprime to $N$, define $f(x) = a^x \mod N$. This function is periodic: there exists a smallest positive integer $r$, called the order of $a$ modulo $N$, such that $a^r \equiv 1 \pmod{N}$, and hence $f(x + r) = f(x)$ for all $x$.

If $r$ is even and $a^{r/2} \not\equiv -1 \pmod{N}$, then since $a^r - 1 = (a^{r/2}-1)(a^{r/2}+1) \equiv 0 \pmod{N}$ but neither factor is itself a multiple of $N$, both

\begin{align*}
\gcd(a^{r/2} - 1,\ N) \qquad \text{and} \qquad \gcd(a^{r/2} + 1,\ N)
\end{align*}

are non-trivial factors of $N$. If the conditions on $r$ fail, one picks a new $a$ and repeats; a randomly chosen $a$ satisfies the conditions with probability at least $1/2$.

To find $r$ on a quantum computer we define the modular exponentiation unitary $U\vert x \rangle = \vert ax \mod N \rangle$. 

$\textbf{The modular exponentiation gate U is unitary.}$

$\textbf{Proof:}$ Since $\gcd(a, N) = 1$, multiplication by $a$ modulo $N$ is a bijection on $\{0, 1, \ldots, N-1\}$. Therefore $U$ acts by permuting the computational basis states, which means its matrix is a permutation matrix. Permutation matrices are unitary: their columns are a permutation of the standard basis vectors, which form an orthonormal set, and a matrix with orthonormal columns satisfies $U^\dagger U = I$.

Its eigenvectors are

\begin{align*}
\vert u_s \rangle = \frac{1}{\sqrt{r}}\sum_{j=0}^{r-1} e^{-2\pi i sj/r}\vert a^j \mod N \rangle, \qquad s = 0, 1, \ldots, r-1,
\end{align*}

with eigenvalues $e^{2\pi i s/r}$. These are precisely the $r$-th roots of unity on the unit circle. QPE applied to $U$ estimates the phase $s/r$, from which $r$ is extracted by classical post-processing.

### **Classical pre-processing to check for easy solutions (reduction)**

If $N$ is even we can instantly factor it into $N = 2 \cdot \frac{N}{2}$. Also we can check if $N$ is a prime power efficiently by using classical approximating algorithms to calculate $N^{1/k}$ for each $k \in \{1, \cdots, i\}$ where $2^i < N < 2^{i+1} $ to see if it is close to a prime integer. For this paper, we will assume we have already checked for these two cases, and that $N$ is neither even nor a prime power. 

Our method (Shor's algorithm) will split any $N$ into two factors $N=pq$. Then we can apply this method recursively to $p$ and to $q$ until we are left only with prime factors. In the application of RSA encryption codebreaking, $N$ would be large and semiprime, so neither pre-processing nor repeat applications of Shor's are necessary.

$\textbf{Updated problem:}$ Given semiprime $N \in \mathbb{N}$, find the prime factorization $N = pq$.

### **Quantum period finding**

The quantum circuit uses two registers. The first register (ancilla) has $t$ qubits, where $t = 2n$ and $n = \lceil \log_2 N \rceil$, chosen to give sufficient precision in the phase estimate. The second register has $n$ qubits and is initialized to $\vert 1 \rangle$.

Although $\vert 1 \rangle$ is not itself an eigenvector of $U$, it is an equal superposition of all $r$ eigenvectors:

\begin{align*}
\vert 1 \rangle = \frac{1}{\sqrt{r}}\sum_{s=0}^{r-1}\vert u_s \rangle.
\end{align*}

This follows from the definition of $\vert u_s \rangle$ and the orthogonality of roots of unity, and means we do not need to know or prepare the eigenvectors explicitly.

QPE is then applied with $U$ as the unitary of interest. By linearity, the algorithm processes all $\vert u_s \rangle$ components simultaneously in superposition. Measurement of the ancilla yields a $t$-bit approximation to $s/r$ for a uniformly random $s \in \{0, \ldots, r-1\}$.



### **Classical post-processing (Euclidean algorithm)**

After measuring the ancilla we obtain an integer $m \in \{0, \ldots, 2^t - 1\}$, a $t$-bit approximation to $2^t \cdot s/r$. Dividing by $2^t$ gives a decimal approximation $\phi \approx s/r$.

To extract $r$ we apply the euclidean algorithm to $\phi$ using a classical computer. Since $r < N$ and $|\phi - s/r| < 1/2^t$, the algorithm recovers $s/r$ exactly as a fraction in lowest terms, provided $t \geq 2\log_2 N$. The denominator is a candidate for $r$ (or a divisor of $r$, in which case small multiples are tried).

$\textbf{Example:}$ Take $N = 15$, $a = 7$. The order is $r = 4$ since $7^4 = 2401 \equiv 1 \pmod{15}$. QPE might return $\phi = 0.25 = 1/4$, giving denominator $4 = r$. Then $a^{r/2} = 7^2 = 49 \equiv 4 \pmod{15}$, and

\begin{align*}
\gcd(4 - 1,\ 15) = \gcd(3,\ 15) = 3, \qquad \gcd(4 + 1,\ 15) = \gcd(5,\ 15) = 5,
\end{align*}

recovering $15 = 3 \times 5$.

### **Quantum circuit for Shor's algorithm**

The quantum circuit for Shor's algorithm is the QPE circuit with $U\vert x\rangle = \vert ax \mod N\rangle$ as the unitary. The first register has $t = 2n$ ancilla qubits; the second has $n = \lceil \log_2 N \rceil$ qubits initialized to $\vert 1 \rangle$.

The circuit proceeds as follows:


- Apply a Hadamard gate to each ancilla qubit, putting the first register into uniform superposition.
- For $k = 0, 1, \ldots, t-1$: apply $C(U^{2^k})$, controlled by the $k$-th ancilla qubit, to the second register.
- Apply $F_{2^t}^\dagger$ (inverse QFT) to the first register.
- Measure the first register to obtain an approximation to $s/r$.


The controlled-$U^{2^k}$ gates are implemented via repeated modular squaring, which can be carried out in $O(n^3)$ elementary gates. This keeps the total circuit complexity polynomial in $n = \log_2 N$, which is the source of Shor's exponential speedup over classical factoring algorithms.

## **Conclusion**

As mentioned, the level of accuracy of the QPE can be increased by using more qubits. The error can be bounded using statistical methods and numerical analysis. This is not addressed in this paper.

One limitation on the usefulness of Shor's algorithm, at least for the present moment, is inavailability of physical quantum computers with high amounts of logical (error resistant) qubits.

## **Bibliography**

### Sources
- Prof. Nievergelt Course notes from math 481
- Barros et al (2026) "a review on QFT"
- Shor (1997) "Polynomial-Time Algorithms for Prime Factorization and Discrete Logarithms on a Quantum Computer"
- Asfaw youtube lectures "Shor's Algorithm I: (parts 1-3)"
